In [2]:
import sys
print(sys.version)

3.10.8 (tags/v3.10.8:aaaf517, Oct 11 2022, 16:50:30) [MSC v.1933 64 bit (AMD64)]


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test").getOrCreate()

data = [("Soham", 21), ("Rahul", 22)]
df = spark.createDataFrame(data, ["Name", "Age"])

df.show()

+-----+---+
| Name|Age|
+-----+---+
|Soham| 21|
|Rahul| 22|
+-----+---+



In [4]:
base_path = "C:/Users/hp/OneDrive/Desktop/pyspark_project/Data/"

In [5]:
def load_csv(file):
    return spark.read.csv(base_path + file, header=True, inferSchema=True)

orders = load_csv("olist_orders_dataset.csv")
customers = load_csv("olist_customers_dataset.csv")
order_items = load_csv("olist_order_items_dataset.csv")
payments = load_csv("olist_order_payments_dataset.csv")
products = load_csv("olist_products_dataset.csv")
sellers = load_csv("olist_sellers_dataset.csv")
reviews = load_csv("olist_order_reviews_dataset.csv")

In [6]:
orders.show(5)
orders.printSchema()
orders.count()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

99441

In [7]:
orders = orders.dropDuplicates()
customers = customers.dropDuplicates()
order_items = order_items.dropDuplicates()
payments = payments.dropDuplicates()
reviews = reviews.dropDuplicates()

In [8]:
from pyspark.sql.functions import to_timestamp, col

orders = orders.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"))
)

orders = orders.withColumn(
    "order_delivered_customer_date",
    to_timestamp(col("order_delivered_customer_date"))
)

orders = orders.withColumn(
    "order_estimated_delivery_date",
    to_timestamp(col("order_estimated_delivery_date"))
)

In [9]:
orders = orders.fillna({"order_status": "unknown"})
reviews = reviews.fillna({"review_score": 0})

In [10]:
orders.selectExpr(
    "count(*) as total_rows",
    "count(order_id) as valid_orders"
).show()

+----------+------------+
|total_rows|valid_orders|
+----------+------------+
|     99441|       99441|
+----------+------------+



In [11]:
df = orders \
    .join(customers, "customer_id", "left") \
    .join(order_items, "order_id", "left") \
    .join(payments, "order_id", "left") \
    .join(products, "product_id", "left") \
    .join(sellers, "seller_id", "left") \
    .join(reviews, "order_id", "left")

In [12]:
df.count()
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = false)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- pa

In [13]:
from pyspark.sql.functions import datediff

df = df.withColumn(
    "delivery_delay_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
)

In [14]:
from pyspark.sql.functions import when

df = df.withColumn(
    "is_delivered",
    when(col("order_status") == "delivered", 1).otherwise(0)
)

In [15]:
from pyspark.sql.functions import year, month

df = df.withColumn("year", year("order_purchase_timestamp"))
df = df.withColumn("month", month("order_purchase_timestamp"))

In [16]:
df = df.withColumn("payment_value", col("payment_value").cast("double"))

In [17]:
df.columns

['order_id',
 'seller_id',
 'product_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_item_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'review_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp',
 'delivery_delay_days',
 'is_delivered',
 'year',
 'month']

In [18]:
df.select(
    "order_id",
    "order_status",
    "delivery_delay_days",
    "is_delivered"
).show(10, False)

+--------------------------------+------------+-------------------+------------+
|order_id                        |order_status|delivery_delay_days|is_delivered|
+--------------------------------+------------+-------------------+------------+
|0c91c4102d1858f4ced29970f55e54a0|delivered   |-21                |1           |
|118045506e1c1dda060171af43fe11b4|delivered   |7                  |1           |
|118045506e1c1dda060171af43fe11b4|delivered   |7                  |1           |
|118045506e1c1dda060171af43fe11b4|delivered   |7                  |1           |
|118045506e1c1dda060171af43fe11b4|delivered   |7                  |1           |
|118045506e1c1dda060171af43fe11b4|delivered   |7                  |1           |
|4380bb6c7d8375926bbea2c8b514dc8c|delivered   |-7                 |1           |
|4aae1a10a22651fe5b76379eb4b34d12|delivered   |1                  |1           |
|69bfec47c3230907ad9552f2f1e0c690|delivered   |-9                 |1           |
|70bd1eb0a425533fa100afb2a4f

In [19]:
df.groupBy("order_status", "is_delivered").count().show()

+------------+------------+------+
|order_status|is_delivered| count|
+------------+------------+------+
|     shipped|           0|  1256|
|    invoiced|           0|   378|
|   delivered|           1|115723|
| unavailable|           0|   652|
|    canceled|           0|   750|
|  processing|           0|   376|
|     created|           0|     5|
|    approved|           0|     3|
+------------+------------+------+



In [20]:
from pyspark.sql.functions import col

df.select([
    col(c).isNull().alias(c)
    for c in ["delivery_delay_days", "is_delivered"]
]).show()

+-------------------+------------+
|delivery_delay_days|is_delivered|
+-------------------+------------+
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
|              false|       false|
+-------------------+------------+
only showing top 20 rows


In [21]:
df.describe(["delivery_delay_days", "payment_value"]).show()

+-------+-------------------+------------------+
|summary|delivery_delay_days|     payment_value|
+-------+-------------------+------------------+
|  count|             115722|            119140|
|   mean| -12.04839183560602|172.73513521906978|
| stddev| 10.163800920097556|267.77607672998346|
|    min|               -147|               0.0|
|    max|                188|          13664.08|
+-------+-------------------+------------------+



In [22]:
df.groupBy("is_delivered").count().show()

+------------+------+
|is_delivered| count|
+------------+------+
|           1|115723|
|           0|  3420|
+------------+------+



In [23]:
df.select(
    "payment_value",
    "delivery_delay_days",
    "is_delivered"
).summary().show()

+-------+------------------+-------------------+------------------+
|summary|     payment_value|delivery_delay_days|      is_delivered|
+-------+------------------+-------------------+------------------+
|  count|            119140|             115722|            119143|
|   mean|172.73513521906978| -12.04839183560602|0.9712949984472441|
| stddev|267.77607672998346| 10.163800920097556|0.1669768201089312|
|    min|               0.0|               -147|                 0|
|    25%|             60.85|                -17|                 1|
|    50%|            108.12|                -13|                 1|
|    75%|            189.19|                 -7|                 1|
|    max|          13664.08|                188|                 1|
+-------+------------------+-------------------+------------------+



In [24]:
df.selectExpr("sum(payment_value) as total_revenue").show()

+-------------------+
|      total_revenue|
+-------------------+
|2.057966400999997E7|
+-------------------+



In [25]:
df.select("order_id").distinct().count()

99441

In [26]:
df.selectExpr("avg(payment_value) as avg_order_value").show()

+------------------+
|   avg_order_value|
+------------------+
|172.73513521906978|
+------------------+



In [27]:
import os
print(os.environ.get("HADOOP_HOME"))

C:\hadoop


In [28]:
spark.version

'4.1.1'

In [29]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test").getOrCreate()

base_path = "C:/Users/hp/OneDrive/Desktop/pyspark_project/Data/"

df = spark.read.csv(
    base_path + "olist_orders_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [30]:
df.limit(100000).toPandas().to_csv(
    base_path + "output/fact_orders.csv",
    index=False
)

PySparkImportError: [PACKAGE_NOT_INSTALLED] Pandas >= 2.2.0 must be installed; however, it was not found.